In [7]:
# ============================================================
# 01_data_collection.ipynb
# Сбор данных: Spotify + MusicBrainz + Deezer
# ============================================================
# Время работы: ~20-40 минут (в основном Deezer)
# Зависимости: kagglehub, musicbrainzngs, requests, gdown, tqdm, orjson
# ============================================================

# @title 1.1. Установка зависимостей
!pip install musicbrainzngs requests gdown kagglehub tqdm orjson -q

# @title 1.2. Импорты и настройки
import os
import json
import time
import datetime
import numpy as np
import pandas as pd
import requests
from tqdm.notebook import tqdm
import musicbrainzngs as mb
import gdown
import orjson  # ⚡ быстрый JSON

# Создать папки
os.makedirs('data', exist_ok=True)
os.makedirs('results/figures', exist_ok=True)

# MusicBrainz
mb.set_useragent("ColabDataAnalysisApp", "1.0", "gareevaazalia12@gmail.com")

# Воспроизводимость
np.random.seed(42)
pd.set_option('display.max_columns', None)

print("✅ Настройки готовы")

# ============================================================
# БЛОК A: SPOTIFY DATASET
# ============================================================

# @title A.1. Загрузка Spotify-датасета
import kagglehub

path = kagglehub.dataset_download(
    "ambaliyagati/spotify-dataset-for-playing-around-with-sql"
)
print("Path to dataset files:", path)

df_path = os.path.join(path, 'spotify_tracks.csv')
if not os.path.exists(df_path):
    for f in os.listdir(path):
        if f.endswith('.csv'):
            df_path = os.path.join(path, f)
            break

df = pd.read_csv(df_path)

print(f"\n✅ Spotify загружен: {df.shape}")
print(df.info())
print(df.head())

# ============================================================
# БЛОК B: MUSICBRAINZ
# ============================================================

# @title B.1. Загрузка кэшей из Google Drive
# --- MusicBrainz album cache ---
try:
    gdown.download(
        url="https://drive.google.com/uc?id=11avDywD_Loc29yk8LUwogrh6MCY5XPVM",
        output="musicbrainz_cache.json", quiet=False
    )
    print("✅ musicbrainz_cache.json загружен")
except Exception as e:
    print(f"⚠️ Ошибка загрузки album-кэша: {e}")

# --- MusicBrainz artist cache ---
try:
    gdown.download(
        url="https://drive.google.com/uc?id=1wHxeDoYIbW6HP7KEsMjJEtfVaex22Rqy",
        output="musicbrainz_artist_cache.json", quiet=False
    )
    print("✅ musicbrainz_artist_cache.json загружен")
except Exception as e:
    print(f"⚠️ Ошибка загрузки artist-кэша: {e}")

# --- Deezer cache ---
try:
    gdown.download(
        url="https://drive.google.com/uc?id=1HAfmc6veh63GychDyP9hpH1crbFuqB1l",
        output="deezer_cache.json", quiet=False
    )
    print("✅ deezer_cache.json загружен")
except Exception as e:
    print(f"⚠️ Ошибка загрузки deezer-кэша: {e}")

# @title B.2. Функции для альбомов MusicBrainz
CACHE_FILE = 'musicbrainz_cache.json'

def load_cache():
    if os.path.exists(CACHE_FILE):
        with open(CACHE_FILE, 'r') as f:
            loaded = json.load(f)
            return {
                k: (v[0], v[1], v[2],
                    (np.nan if v[3] is None else v[3]), v[4])
                for k, v in loaded.items()
            }
    return {}

def save_cache(cache_data):
    serializable = {
        k: (v[0], v[1], v[2],
            (None if pd.isna(v[3]) else v[3]), v[4])
        for k, v in cache_data.items()
    }
    with open(CACHE_FILE, 'w') as f:
        json.dump(serializable, f, indent=4)

def get_musicbrainz_album_info(artist_name, album_title, cache):
    cache_key = f"{artist_name}|{album_title}"
    if cache_key in cache:
        return cache[cache_key]

    retries = 3
    for attempt in range(retries):
        try:
            time.sleep(1)
            result = mb.search_releases(artist=artist_name,
                                        release=album_title, limit=1)

            country, status, label, year = 'unknown', 'unknown', 'unknown', np.nan

            if result and result.get('release-list'):
                release = result['release-list'][0]

                country_val = release.get('country')
                if country_val and str(country_val).strip():
                    country = str(country_val).strip()

                status_val = release.get('status')
                if status_val and str(status_val).strip():
                    status = str(status_val).strip()

                label_info = release.get('label-info')
                if label_info:
                    for li in label_info:
                        lbl = li.get('label') or {}
                        lbl_name = lbl.get('name')
                        if lbl_name and str(lbl_name).strip():
                            label = str(lbl_name).strip()
                            break

                date_val = release.get('date')
                if date_val:
                    try:
                        year = int(str(date_val).split('-')[0])
                    except (ValueError, IndexError):
                        pass

            has_data = 1 if (
                country != 'unknown' or
                status != 'unknown' or
                label != 'unknown' or
                not pd.isna(year)
            ) else 0

            cache[cache_key] = (country, status, label, year, has_data)
            return country, status, label, year, has_data

        except mb.WebServiceError as e:
            print(f"Ошибка API: {e}")
            if attempt < retries - 1:
                time.sleep(5)
            else:
                cache[cache_key] = ('unknown', 'unknown', 'unknown', np.nan, 0)
                return 'unknown', 'unknown', 'unknown', np.nan, 0
        except Exception as e:
            print(f"Неожиданная ошибка: {e}")
            cache[cache_key] = ('unknown', 'unknown', 'unknown', np.nan, 0)
            return 'unknown', 'unknown', 'unknown', np.nan, 0

    cache[cache_key] = ('unknown', 'unknown', 'unknown', np.nan, 0)
    return 'unknown', 'unknown', 'unknown', np.nan, 0

# @title B.3. Обогащение альбомов
for col, default in [
    ('release_country', 'unknown'),
    ('release_status', 'unknown'),
    ('release_label', 'unknown'),
    ('album_year', np.nan),
    ('has_musicbrainz_data', 0),
]:
    if col not in df.columns:
        df[col] = default

unique_albums_df = df[['artists', 'album']].drop_duplicates().copy()
musicbrainz_album_cache = load_cache()
print(f"Загружено {len(musicbrainz_album_cache)} записей из кэша альбомов.")
print(f"Загрузка данных для {len(unique_albums_df)} уникальных пар...")

unique_albums_df['mb_release_country'] = 'unknown'
unique_albums_df['mb_release_status'] = 'unknown'
unique_albums_df['mb_release_label'] = 'unknown'
unique_albums_df['mb_album_year'] = np.nan
unique_albums_df['mb_has_data'] = 0

processed = 0
for index, row in tqdm(unique_albums_df.iterrows(),
                       total=len(unique_albums_df), desc="Альбомы"):
    country, status, label, year, has_data = get_musicbrainz_album_info(
        row['artists'], row['album'], musicbrainz_album_cache
    )
    unique_albums_df.loc[index, 'mb_release_country'] = country
    unique_albums_df.loc[index, 'mb_release_status'] = status
    unique_albums_df.loc[index, 'mb_release_label'] = label
    unique_albums_df.loc[index, 'mb_album_year'] = year
    unique_albums_df.loc[index, 'mb_has_data'] = has_data

    processed += 1
    if processed % 500 == 0:  # ⚡ реже сохраняем
        save_cache(musicbrainz_album_cache)

save_cache(musicbrainz_album_cache)
print(f"✅ Обработано {processed} альбомов")

# @title B.4. Индикатор полноты альбома
unique_albums_df['album_data_completeness'] = (
    (unique_albums_df['mb_release_country'] != 'unknown').astype(int) +
    (unique_albums_df['mb_release_label'] != 'unknown').astype(int) +
    (unique_albums_df['mb_album_year'].notna()).astype(int)
) / 3.0

# @title B.5. Слияние данных альбомов
df = df.merge(
    unique_albums_df[[
        'artists', 'album',
        'mb_release_country', 'mb_release_status', 'mb_release_label',
        'mb_album_year', 'mb_has_data', 'album_data_completeness'
    ]],
    on=['artists', 'album'], how='left'
)

df['release_country'] = np.where(
    (df['mb_has_data'] == 1) & (df['mb_release_country'] != 'unknown'),
    df['mb_release_country'], df['release_country']
)
df['release_status'] = np.where(
    (df['mb_has_data'] == 1) & (df['mb_release_status'] != 'unknown'),
    df['mb_release_status'], df['release_status']
)
df['release_label'] = np.where(
    (df['mb_has_data'] == 1) & (df['mb_release_label'] != 'unknown'),
    df['mb_release_label'], df['release_label']
)
df['album_year'] = np.where(
    (df['mb_has_data'] == 1) & (df['mb_album_year'].notna()),
    df['mb_album_year'], df['album_year']
)

df = df.drop(columns=[
    'mb_release_country', 'mb_release_status', 'mb_release_label',
    'mb_album_year', 'mb_has_data'
])

# @title B.6. Функции для артистов MusicBrainz
ARTIST_CACHE_FILE = 'musicbrainz_artist_cache.json'

def load_artist_cache():
    if os.path.exists(ARTIST_CACHE_FILE):
        with open(ARTIST_CACHE_FILE, 'r') as f:
            loaded = json.load(f)
            return {
                k: (v[0], v[1],
                    (np.nan if v[2] is None else v[2]),
                    (np.nan if v[3] is None else v[3]),
                    v[4], v[5])
                for k, v in loaded.items()
            }
    return {}

def save_artist_cache(cache_data):
    serializable = {
        k: (v[0], v[1],
            (None if pd.isna(v[2]) else v[2]),
            (None if pd.isna(v[3]) else v[3]),
            v[4], v[5])
        for k, v in cache_data.items()
    }
    with open(ARTIST_CACHE_FILE, 'w') as f:
        json.dump(serializable, f, indent=4)

def get_musicbrainz_artist_details(artist_name, artist_cache):
    cache_key = artist_name
    if cache_key in artist_cache:
        return artist_cache[cache_key]

    primary_artist = artist_name.split(',')[0].strip()

    retries = 3
    for attempt in range(retries):
        try:
            time.sleep(1)
            result = mb.search_artists(artist=primary_artist, limit=1)

            artist_type, country = 'unknown', 'unknown'
            begin_year, end_year = np.nan, np.nan
            is_active = False

            if result and result.get('artist-list'):
                artist = result['artist-list'][0]

                type_val = artist.get('type')
                if type_val and str(type_val).strip():
                    artist_type = str(type_val).strip()

                country_val = artist.get('country')
                if country_val and str(country_val).strip():
                    country = str(country_val).strip()

                life_span = artist.get('life-span') or {}

                begin_val = life_span.get('begin')
                if begin_val:
                    try:
                        begin_year = int(str(begin_val).split('-')[0])
                    except (ValueError, IndexError):
                        pass

                end_val = life_span.get('end')
                if end_val:
                    try:
                        end_year = int(str(end_val).split('-')[0])
                    except (ValueError, IndexError):
                        pass

                if 'ended' in life_span:
                    is_active = not bool(life_span['ended'])
                else:
                    is_active = pd.isna(end_year)

            has_data = 1 if (
                artist_type != 'unknown' or
                country != 'unknown' or
                not pd.isna(begin_year) or
                not pd.isna(end_year)
            ) else 0

            artist_cache[cache_key] = (artist_type, country, begin_year,
                                       end_year, is_active, has_data)
            return artist_type, country, begin_year, end_year, is_active, has_data

        except mb.WebServiceError as e:
            print(f"Ошибка API артиста {primary_artist}: {e}")
            if attempt < retries - 1:
                time.sleep(5)
            else:
                artist_cache[cache_key] = ('unknown', 'unknown', np.nan,
                                           np.nan, False, 0)
                return 'unknown', 'unknown', np.nan, np.nan, False, 0
        except Exception as e:
            print(f"Ошибка артиста: {e}")
            artist_cache[cache_key] = ('unknown', 'unknown', np.nan,
                                       np.nan, False, 0)
            return 'unknown', 'unknown', np.nan, np.nan, False, 0

    artist_cache[cache_key] = ('unknown', 'unknown', np.nan, np.nan, False, 0)
    return 'unknown', 'unknown', np.nan, np.nan, False, 0

# @title B.7. Обогащение артистов
for col, default in [
    ('artist_type', 'unknown'),
    ('artist_country', 'unknown'),
    ('artist_begin_year', np.nan),
    ('artist_end_year', np.nan),
    ('artist_is_active', False),
    ('has_musicbrainz_artist_data', 0),
]:
    if col not in df.columns:
        df[col] = default

unique_artists_df = df[['artists']].drop_duplicates().copy()
unique_artists_df['mb_artist_type'] = 'unknown'
unique_artists_df['mb_artist_country'] = 'unknown'
unique_artists_df['mb_artist_begin_year'] = np.nan
unique_artists_df['mb_artist_end_year'] = np.nan
unique_artists_df['mb_artist_is_active'] = False
unique_artists_df['mb_has_artist_data'] = 0

artist_cache = load_artist_cache()
print(f"Загружено {len(artist_cache)} записей из кэша артистов.")
print(f"Загрузка данных для {len(unique_artists_df)} артистов...")

processed_artists = 0
for index, row in tqdm(unique_artists_df.iterrows(),
                       total=len(unique_artists_df), desc="Артисты"):
    artist_type, country, begin_year, end_year, is_active, has_data = \
        get_musicbrainz_artist_details(row['artists'], artist_cache)

    unique_artists_df.loc[index, 'mb_artist_type'] = artist_type
    unique_artists_df.loc[index, 'mb_artist_country'] = country
    unique_artists_df.loc[index, 'mb_artist_begin_year'] = begin_year
    unique_artists_df.loc[index, 'mb_artist_end_year'] = end_year
    unique_artists_df.loc[index, 'mb_artist_is_active'] = is_active
    unique_artists_df.loc[index, 'mb_has_artist_data'] = has_data

    processed_artists += 1
    if processed_artists % 500 == 0:  # ⚡ реже сохраняем
        save_artist_cache(artist_cache)

save_artist_cache(artist_cache)
print(f"✅ Обработано {processed_artists} артистов")

# @title B.8. Индикатор полноты артиста
unique_artists_df['artist_data_completeness'] = (
    (unique_artists_df['mb_artist_type'] != 'unknown').astype(int) +
    (unique_artists_df['mb_artist_country'] != 'unknown').astype(int) +
    (unique_artists_df['mb_artist_begin_year'].notna()).astype(int)
) / 3.0

# @title B.9. Слияние данных артистов
df = df.merge(
    unique_artists_df[[
        'artists',
        'mb_artist_type', 'mb_artist_country',
        'mb_artist_begin_year', 'mb_artist_end_year',
        'mb_artist_is_active', 'mb_has_artist_data',
        'artist_data_completeness'
    ]],
    on='artists', how='left'
)

df['artist_type'] = np.where(
    (df['mb_has_artist_data'] == 1) & (df['mb_artist_type'] != 'unknown'),
    df['mb_artist_type'], df['artist_type']
)
df['artist_country'] = np.where(
    (df['mb_has_artist_data'] == 1) & (df['mb_artist_country'] != 'unknown'),
    df['mb_artist_country'], df['artist_country']
)
df['artist_begin_year'] = np.where(
    (df['mb_has_artist_data'] == 1) & (df['mb_artist_begin_year'].notna()),
    df['mb_artist_begin_year'], df['artist_begin_year']
)
df['artist_end_year'] = np.where(
    (df['mb_has_artist_data'] == 1) & (df['mb_artist_end_year'].notna()),
    df['mb_artist_end_year'], df['artist_end_year']
)
df['artist_is_active'] = np.where(
    (df['mb_has_artist_data'] == 1),
    df['mb_artist_is_active'], df['artist_is_active']
)

df = df.drop(columns=[
    'mb_artist_type', 'mb_artist_country',
    'mb_artist_begin_year', 'mb_artist_end_year',
    'mb_artist_is_active', 'mb_has_artist_data'
])

# @title B.10. Пересчёт флагов + career duration
if 'album_data_completeness' not in df.columns:
    df['album_data_completeness'] = 0.0
if 'artist_data_completeness' not in df.columns:
    df['artist_data_completeness'] = 0.0

df['has_musicbrainz_data'] = np.where(
    df['album_data_completeness'] > 0, 1, 0
)
df['has_musicbrainz_artist_data'] = np.where(
    df['artist_data_completeness'] > 0, 1, 0
)

current_year = pd.Timestamp.now().year
df['artist_career_duration_years'] = np.where(
    df['has_musicbrainz_artist_data'] == 1,
    np.where(
        df['artist_type'] == 'Group',
        np.where(
            df['artist_begin_year'].notna(),
            df['artist_end_year'].fillna(current_year) - df['artist_begin_year'],
            np.nan
        ),
        np.nan
    ),
    np.nan
)
df['artist_career_duration_years'] = df['artist_career_duration_years'].replace(0, np.nan)

print(f"✅ MusicBrainz завершён: {df.shape}")

# ============================================================
# БЛОК C: DEEZER (ОПТИМИЗИРОВАННЫЙ)
# ============================================================

# @title C.1. Функции очистки + Deezer API (с orjson)
import re

DEEZER_CACHE_FILE = "deezer_cache.json"
RATE_LIMIT_DELAY = 0.2

NOISE_PATTERN = re.compile(
    r'''
    \s*[\(\-–—\[]\s*
    (?:
        acoustic | instrumental |
        remaster(?:ed)?(?:\s*\d{4})? |
        live(?:\s*(?:at|from|version))? |
        demo | radio\s*edit | single\s*version | album\s*version |
        deluxe(?:\s*edition)? | bonus\s*track |
        feat\.?.* | featuring.* | with\s+.* | ft\.?.* |
        explicit | clean | mono | stereo |
        version | edit | mix | remix | reprise | take\s*\d+ |
        session | rehearsal | karaoke | cover | tribute |
        instrumental\s*version | spanish\s*version | french\s*version |
        english\s*version | extended |
        original\s*mix | album\s*mix | radio\s*mix | club\s*mix |
        \d{4}\s*remaster(?:ed)? | \d{4}\s*version | \d{4}\s*mix
    )
    (?:\s*[\)\]])?\s*$
    ''',
    re.IGNORECASE | re.VERBOSE
)
TRAILING_PARENS = re.compile(r'\s*\([^)]*\)\s*$')
TRAILING_BRACKETS = re.compile(r'\s*\[[^\]]*\]\s*$')

def clean_track_name(name):
    if not isinstance(name, str):
        return name
    original = name
    cleaned = name.strip()
    cleaned = NOISE_PATTERN.sub('', cleaned).strip()
    prev = None
    while prev != cleaned:
        prev = cleaned
        cleaned = TRAILING_PARENS.sub('', cleaned).strip()
        cleaned = TRAILING_BRACKETS.sub('', cleaned).strip()
    cleaned = re.sub(r'\s*[\-–—]\s*$', '', cleaned).strip()
    if not cleaned:
        return original
    return cleaned

def load_deezer_cache():
    if os.path.exists(DEEZER_CACHE_FILE):
        with open(DEEZER_CACHE_FILE, 'rb') as f:
            return orjson.loads(f.read())
    return {}

def save_deezer_cache(cache):
    # ⚡ orjson в 3-10 раз быстрее стандартного json
    with open(DEEZER_CACHE_FILE, 'wb') as f:
        f.write(orjson.dumps(cache))

def deezer_search_track(track_name, artist_name, cache):
    """Поиск трека. Sleep ТОЛЬКО при сетевом запросе."""
    primary_artist = artist_name.split(",")[0].strip()
    cleaned_name = clean_track_name(track_name)

    candidates = [cleaned_name]
    if cleaned_name != track_name:
        candidates.append(track_name)

    for candidate in candidates:
        cache_key = f"search|{candidate}|{primary_artist}"
        if cache_key in cache:
            cached = cache[cache_key]
            if cached is not None:
                return cached, True   # ✅ из кэша, БЕЗ sleep
            continue

        query = f'{candidate} {primary_artist}'
        try:
            url = "https://api.deezer.com/search"
            params = {"q": query, "limit": 1}
            response = requests.get(url, params=params, timeout=10)
            time.sleep(RATE_LIMIT_DELAY)   # ✅ sleep ТОЛЬКО здесь

            if response.status_code == 200:
                data = response.json()
                if data.get("data") and len(data["data"]) > 0:
                    track_id = data["data"][0]["id"]
                    cache[cache_key] = track_id
                    return track_id, False

            cache[cache_key] = None
        except Exception as e:
            print(f"Ошибка поиска {candidate}: {e}")
            cache[cache_key] = None

    return None, False

def deezer_get_track_features(track_id, cache):
    """Возвращает (dict | None, from_cache: bool)."""
    cache_key = f"track|{track_id}"
    if cache_key in cache:
        return cache[cache_key], True   # ✅ из кэша, БЕЗ sleep

    try:
        url = f"https://api.deezer.com/track/{track_id}"
        response = requests.get(url, timeout=10)
        time.sleep(RATE_LIMIT_DELAY)

        if response.status_code == 200:
            data = response.json()
            result = {
                "deezer_bpm": data.get("bpm"),
                "deezer_gain": data.get("gain"),
                "deezer_rank": data.get("rank"),
                "deezer_duration": data.get("duration"),
                "deezer_explicit": int(bool(data.get("explicit_lyrics"))),
                "deezer_artist_id": data.get("artist", {}).get("id"),
            }
            cache[cache_key] = result
            return result, False

        cache[cache_key] = None
        return None, False
    except Exception as e:
        print(f"Ошибка трека {track_id}: {e}")
        cache[cache_key] = None
        return None, False

def deezer_get_artist_features(artist_id, cache):
    """Возвращает (dict | None, from_cache: bool)."""
    cache_key = f"artist|{artist_id}"
    if cache_key in cache:
        return cache[cache_key], True   # ✅ из кэша, БЕЗ sleep

    try:
        url = f"https://api.deezer.com/artist/{artist_id}"
        response = requests.get(url, timeout=10)
        time.sleep(RATE_LIMIT_DELAY)

        if response.status_code == 200:
            data = response.json()
            result = {
                "deezer_artist_fans": data.get("nb_fan"),
                "deezer_artist_albums": data.get("nb_album"),
            }
            cache[cache_key] = result
            return result, False

        cache[cache_key] = None
        return None, False
    except Exception as e:
        print(f"Ошибка артиста {artist_id}: {e}")
        cache[cache_key] = None
        return None, False

# @title C.2. Обогащение Deezer
deezer_columns = [
    "deezer_bpm", "deezer_gain", "deezer_rank",
    "deezer_duration", "deezer_explicit",
    "deezer_artist_fans", "deezer_artist_albums",
    "deezer_found",
]
for col in deezer_columns:
    if col not in df.columns:
        df[col] = 0 if col in ("deezer_found", "deezer_explicit") else None

if "deezer_artist_id" not in df.columns:
    df["deezer_artist_id"] = None

deezer_cache = load_deezer_cache()
print(f"Загружено {len(deezer_cache)} записей из кэша Deezer")

# Статистика кэша
n_search = sum(1 for k in deezer_cache if k.startswith('search|'))
n_track = sum(1 for k in deezer_cache if k.startswith('track|'))
n_artist = sum(1 for k in deezer_cache if k.startswith('artist|'))
print(f"  search: {n_search}, track: {n_track}, artist: {n_artist}")

processed_d = 0
new_requests = 0

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Deezer"):
    # ✅ Пропускаем уже обработанные (deezer_found = 1)
    if pd.notna(row.get("deezer_found")) and row["deezer_found"] == 1:
        continue

    track_name = str(row["name"])
    artist_name = str(row["artists"])

    # 1. Поиск трека
    track_id, from_cache = deezer_search_track(track_name, artist_name,
                                                deezer_cache)
    if not from_cache:
        new_requests += 1

    if track_id is None:
        df.at[idx, "deezer_found"] = 0
        processed_d += 1
        if processed_d % 500 == 0:      # ⚡ реже сохраняем
            save_deezer_cache(deezer_cache)
        continue

    # 2. Детали трека
    track_features, from_cache = deezer_get_track_features(track_id,
                                                            deezer_cache)
    if not from_cache:
        new_requests += 1

    if track_features:
        for key, value in track_features.items():
            if key in df.columns:
                df.at[idx, key] = value
        df.at[idx, "deezer_found"] = 1

        # 3. Детали артиста
        artist_id = track_features.get("deezer_artist_id")
        if artist_id:
            artist_features, from_cache = deezer_get_artist_features(
                artist_id, deezer_cache
            )
            if not from_cache:
                new_requests += 1

            if artist_features:
                df.at[idx, "deezer_artist_fans"] = \
                    artist_features.get("deezer_artist_fans")
                df.at[idx, "deezer_artist_albums"] = \
                    artist_features.get("deezer_artist_albums")

    processed_d += 1
    if processed_d % 500 == 0:      # ⚡ реже сохраняем
        save_deezer_cache(deezer_cache)

save_deezer_cache(deezer_cache)

# Удалить служебный столбец
df = df.drop(columns=["deezer_artist_id"], errors="ignore")

print(f"\n✅ Deezer завершён: {df.shape}")
print(f"Найдено в Deezer: {df['deezer_found'].sum()} из {len(df)} "
      f"({df['deezer_found'].mean():.1%})")
print(f"Сетевых запросов за прогон: {new_requests}")

# ============================================================
# БЛОК D: СОХРАНЕНИЕ
# ============================================================

# @title D.1. Сохранение сырого датасета
df.to_pickle('data/01_raw_enriched.pkl')
df.to_csv('data/01_raw_enriched.csv', index=False)

print(f"\n{'=' * 60}")
print(f"✅ 01_data_collection ЗАВЕРШЁН")
print(f"{'=' * 60}")
print(f"Размер: {df.shape}")
print(f"deezer_rank заполнен: {df['deezer_rank'].notna().sum()} "
      f"({df['deezer_rank'].notna().mean():.1%})")
print(f"Сохранено: data/01_raw_enriched.pkl")

✅ Настройки готовы
Using Colab cache for faster access to the 'spotify-dataset-for-playing-around-with-sql' dataset.
Path to dataset files: /kaggle/input/spotify-dataset-for-playing-around-with-sql

✅ Spotify загружен: (6300, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6300 entries, 0 to 6299
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   id           6300 non-null   object
 1   name         6300 non-null   object
 2   genre        6300 non-null   object
 3   artists      6300 non-null   object
 4   album        6300 non-null   object
 5   popularity   6300 non-null   int64 
 6   duration_ms  6300 non-null   int64 
 7   explicit     6300 non-null   bool  
dtypes: bool(1), int64(2), object(5)
memory usage: 350.8+ KB
None
                       id                                           name  \
0  7kr3xZk4yb3YSZ4VFtg2Qt                                       Acoustic   
1  1kJygfS4eoVziBBI93MSYp            

Downloading...
From: https://drive.google.com/uc?id=11avDywD_Loc29yk8LUwogrh6MCY5XPVM
To: /content/musicbrainz_cache.json
100%|██████████| 729k/729k [00:00<00:00, 130MB/s]


✅ musicbrainz_cache.json загружен


Downloading...
From: https://drive.google.com/uc?id=1wHxeDoYIbW6HP7KEsMjJEtfVaex22Rqy
To: /content/musicbrainz_artist_cache.json
100%|██████████| 557k/557k [00:00<00:00, 116MB/s]


✅ musicbrainz_artist_cache.json загружен


...

Артисты:   0%|          | 0/4651 [00:00<?, ?it/s]

✅ Обработано 4651 артистов
✅ MusicBrainz завершён: (6300, 22)
Загружено 14206 записей из кэша Deezer
  search: 5922, track: 4738, artist: 3546


Deezer:   0%|          | 0/6300 [00:00<?, ?it/s]

/tmp/ipykernel_2738/1176936341.py:706: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'False' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.at[idx, key] = value



✅ Deezer завершён: (6300, 30)
Найдено в Deezer: 5241 из 6300 (83.2%)
Сетевых запросов за прогон: 2729

✅ 01_data_collection ЗАВЕРШЁН
Размер: (6300, 30)
deezer_rank заполнен: 5241 (83.2%)
Сохранено: data/01_raw_enriched.pkl
